In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from models.unet_3d import UNet3D
from utils.unets_helper_functions import (
    set_seed,
    train_one_epoch,
    validate_one_epoch,
    save_checkpoint,
    PatchDataset,
    evaluate_full_volume
)


In [3]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))

Train cases: 33
Val cases: 9


In [5]:
PATCH_SIZE = 80
PATCHES_PER_CASE = 6

train_dataset = PatchDataset(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        augment=False)

val_dataset = PatchDataset(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=2,    
    pin_memory=True
)

In [6]:
model = UNet3D(
    in_channels=1,
    out_channels=7,
    base_filters=16
).to(device)

print("Model initialized.")

Model initialized.


In [7]:
optimizer = optim.Adam(model.parameters(), lr=1e-4)
weights = torch.tensor([0.1, 1, 1, 1, 1, 1, 1]).to(device)
ce_loss = nn.CrossEntropyLoss(weight=weights)
scaler = torch.cuda.amp.GradScaler()

C:\Users\dhanu\AppData\Local\Temp\ipykernel_6232\1548250201.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [8]:
EPOCHS = 40

PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "unet_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        ce_loss,
        device
    )

    val_loss, val_dice = validate_one_epoch(
        model,
        val_loader,
        ce_loss,
        device
    )

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_model.pth")
        )
        print("Saved Best Model")

  0%|          | 0/99 [00:00<?, ?it/s]

100%|██████████| 99/99 [08:11<00:00,  4.97s/it]



Epoch 0
Train Loss: 2.9405
Val Loss: 2.7977
Per Class Dice: [6.16968465e-02 2.24897451e-06 1.23847832e-03 5.67575315e-03
 1.03658654e-02 1.02101282e-01]
Saved Best Model


100%|██████████| 99/99 [05:41<00:00,  3.45s/it]



Epoch 1
Train Loss: 2.6727
Val Loss: 2.6591
Per Class Dice: [0.04467258 0.11677807 0.00324895 0.05416658 0.00413119 0.13718109]
Saved Best Model


...


Epoch 14
Train Loss: 1.6105
Val Loss: 1.6461
Per Class Dice: [0.47209194 0.82326674 0.55555556 0.38257893 0.2625997  0.3279235 ]


100%|██████████| 99/99 [06:51<00:00,  4.16s/it]



Epoch 15
Train Loss: 1.5684
Val Loss: 1.5801
Per Class Dice: [0.44068764 0.89786326 0.55555556 0.328711   0.36397731 0.3337118 ]
Saved Best Model


 91%|█████████ | 90/99 [06:01<00:36,  4.01s/it]


KeyboardInterrupt: 

In [8]:
#  RESUME TRAINING 
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "unet_fold0")

start_epoch = 0
best_val_loss = float("inf")

resume_path = os.path.join(SAVE_DIR, "best_model.pth")  

if os.path.exists(resume_path):
    print("Loading checkpoint:", resume_path)

    checkpoint = torch.load(resume_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]

    print(f"Resuming from epoch {start_epoch}")
else:
    print("No checkpoint found, starting fresh.")


Loading checkpoint: c:\Users\dhanu\Desktop\mini-project-1\experiments\unet_fold0\best_model.pth
Resuming from epoch 16


C:\Users\dhanu\AppData\Local\Temp\ipykernel_6232\3872517435.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(resume_path, map_location=device)


In [10]:
EPOCHS = 40
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "unet_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        ce_loss,
        device
    )

    val_loss, val_dice = validate_one_epoch(
        model,
        val_loader,
        ce_loss,
        device
    )

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_model.pth")
        )
        print("Saved Best Model")

100%|██████████| 99/99 [05:04<00:00,  3.07s/it]



Epoch 0
Train Loss: 1.5484
Val Loss: 1.5046
Per Class Dice: [0.68387318 0.83034942 0.66666695 0.6676505  0.26061589 0.6391875 ]
Saved Best Model


100%|██████████| 99/99 [04:39<00:00,  2.82s/it]



Epoch 1
Train Loss: 1.4880
Val Loss: 1.4771
Per Class Dice: [0.61233602 0.73231394 0.66666667 0.62847349 0.39454936 0.52752268]
Saved Best Model


100%|██████████| 99/99 [05:31<00:00,  3.35s/it]


...


Epoch 24
Train Loss: 1.0147
Val Loss: 0.9549
Per Class Dice: [0.79493628 0.85621588 0.55555556 0.72787144 0.76251099 0.87643252]
Saved Best Model


100%|██████████| 99/99 [05:39<00:00,  3.43s/it]



Epoch 25
Train Loss: 0.9910
Val Loss: 0.9900
Per Class Dice: [0.4698914  0.93698092 0.66666667 0.62718891 0.61738763 0.4481755 ]


 58%|█████▊    | 57/99 [03:20<02:27,  3.51s/it]


KeyboardInterrupt: 

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet3D().to(device)

model_path = "../experiments/unet_fold0/best_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


Model loaded for testing.


C:\Users\dhanu\AppData\Local\Temp\ipykernel_26616\3740780336.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


In [12]:
mean_dice, std_dice = evaluate_full_volume(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.8768281592122439), np.float64(0.8437210088186547), np.float64(3.752345201679755e-09), np.float64(0.6803811831187557), np.float64(0.7632063586670482), np.float64(0.8866197826492611)]
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.8541621387051287), np.float64(0.7910046020743209), np.float64(3.4340659222731253e-09), np.float64(0.7966350783982997), np.float64(0.7892194255090138), np.float64(0.8584967265779521)]
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.8112759478637509), np.float64(0.7793083657775087), np.float64(3.409478338187936e-09), np.float64(0.7484893400134025), np.float64(0.8342920508460101), np.float64(0.894142520799234)]
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_14 Dice: [np.float64(0.8581276144882672), np.float64(0.7282570974330137), np.float64(4.496402

In [8]:
mean_dice, std_dice = evaluate_full_volume(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Volume: 272x357x357 | Patches: 1521
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.8846944948857364), np.float64(0.8300474889696018), np.float64(3.752345201679755e-09), np.float64(0.7200824221587554), np.float64(0.7860180481422901), np.float64(0.8857292362307195)]
Volume: 243x383x383 | Patches: 1568
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.8613414334177978), np.float64(0.77783259350614), np.float64(3.4340659222731253e-09), np.float64(0.7976704489418263), np.float64(0.8100541281589908), np.float64(0.8742818126010822)]
Volume: 264x333x333 | Patches: 1296
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.8380634391117197), np.float64(0.8159921580899089), np.float64(3.409478338187936e-09), np.float64(0.7476562062832447), np.float64(0.8480318884988028), np.float64(0.9127361113456184)]
Volume: 248x395x395 | Patches: 1800
Unique pred labels: [0 

## UNet 3D Patch-Based Validation Performance

The model was trained for 40 epochs using patch-based validation.  
Below are the final patch validation metrics obtained at the best validation loss epoch.

### Final Training Statistics (Epoch 39 - Best Model)

- Train Loss: 1.0147
- Val Loss: 0.9549

## Observations

- Validation loss steadily improved toward final epochs.
- Model converged well overall.
- Some class instability observed during training.


Train Loss 0.9783 | Val Loss 0.9308

# Evaluating on Full Volume
~ (patch = 80 , stride = 60)

| Organ ID | Organ (Based on Your Map) | Mean Dice   | Std Dev |
| -------- | ------------------------- | ----------- | ------- |
| 1        | Bone Mandible             | **0.8685**  | 0.0306  |
| 2        | Brainstem                 | **0.8169**  | 0.0250  |
| 3        | Spinal Cord               | **~0.0000** | ~0.0000 |
| 4        | Parotid Left              | **0.7135**  | 0.0452  |
| 5        | Parotid Right             | **0.7690**  | 0.0764  |
| 6        | Oral Cavity               | **0.8376**  | 0.0531  |

SpinalCord Fails maybe beacuse It is extremely thin structure
Very small voxel count
Hard to capture with 80³ patches
May not be sufficiently sampled